In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install --no-deps --upgrade timm
import importlib.metadata as _md
try: _tv = tuple(int(x) for x in _md.version("torch").split("+")[0].split(".")[:2])
except Exception: _tv = (2, 11)
_torchcodec = {(2, 11): "torchcodec>=0.11.0", (2, 10): "torchcodec==0.10.*", (2, 9): "torchcodec==0.9.*", (2, 8): "torchcodec==0.7.*", (2, 7): "torchcodec==0.5.*"}.get(_tv, "torchcodec>=0.12.0" if _tv >= (2, 12) else "torchcodec")
!pip install "{_torchcodec}"
import torch; torch._dynamo.config.recompile_limit = 64

In [2]:
from unsloth import FastModel          # must be imported first
import torch, json, random, re, os, glob, gzip, zipfile, urllib.request
import numpy as np, pandas as pd
from datasets import load_dataset, Dataset
from huggingface_hub import hf_hub_download

SEED = 3407
random.seed(SEED); np.random.seed(SEED)

PROMPT        = "Generate synthetic data."
SMOKE_TEST    = True     # True = 30-step pipeline test (~5 min). Set False for the real run.
N_IQA         = 2000     # InsuranceQA records in training mix
N_BITEXT      = 2000     # Bitext records in training mix
BENCH_REPEAT  = 2        # benchmark has only 510 rows, so repeat them
MAX_SEQ_LEN   = 1024
MAX_EX_TOKENS = 900      # drop any example longer than this, so JSON is never cut off

print(torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Tesla T4


In [3]:
if not glob.glob("insuranceQA-master/V2"):
    urllib.request.urlretrieve(
        "https://github.com/shuzi/insuranceQA/archive/refs/heads/master.zip", "iqa.zip")
    zipfile.ZipFile("iqa.zip").extractall(".")
V2 = glob.glob("*/V2")[0]
print("V2 files:", sorted(os.listdir(V2)))

def find(pattern):
    return sorted(p for p in glob.glob(V2 + "/*") if re.search(pattern, os.path.basename(p)))

vocab_p = find(r"^vocabulary")
ans_p   = find(r"label2answer\.token\.encoded")
q_p     = find(r"question\.anslabel\.token\.100\.pool\.solr\.(train|valid|test)\.encoded")
assert vocab_p and ans_p and q_p, f"files not found, see listing above: {vocab_p} {ans_p} {q_p}"

def read_lines(path):
    op = gzip.open if path.endswith(".gz") else open
    with op(path, "rt", encoding="utf-8", errors="ignore") as f:
        return f.read().splitlines()

vocab = {}
for l in read_lines(vocab_p[0]):
    p = l.split("\t")
    if len(p) == 2: vocab[p[0]] = p[1]

unknown = 0
def decode(s):
    global unknown
    out = []
    for t in s.split():
        if t in vocab: out.append(vocab[t])
        else: unknown += 1; out.append(t)
    return " ".join(out)

def detok(s):   # the corpus is tokenized ("word ."), so tidy the spacing
    s = re.sub(r"\s+([.,;:!?%)])", r"\1", s)
    s = re.sub(r"([($])\s+", r"\1", s)
    return s.replace(" 's", "'s").replace(" n't", "n't").strip()

answers = {}
for l in read_lines(ans_p[0]):
    p = l.split("\t")
    if len(p) == 2: answers[p[0]] = detok(decode(p[1]))

iqa = []
for path in q_p:                                  # train + valid + test, all of it
    for l in read_lines(path):
        p = l.split("\t")
        if len(p) < 3: continue
        gt = p[2].split()
        if gt and gt[0] in answers:
            iqa.append({"source": "insuranceqa", "domain": p[0],
                        "question": detok(decode(p[1])), "answer": answers[gt[0]]})
print(len(iqa), "InsuranceQA records | unknown tokens:", unknown)
print(iqa[0])

V2 files: ['InsuranceQA.label2answer.raw.encoded.gz', 'InsuranceQA.label2answer.token.encoded.gz', 'InsuranceQA.question.anslabel.raw.100.pool.solr.test.encoded.gz', 'InsuranceQA.question.anslabel.raw.100.pool.solr.train.encoded.gz', 'InsuranceQA.question.anslabel.raw.100.pool.solr.valid.encoded.gz', 'InsuranceQA.question.anslabel.raw.1000.pool.solr.test.encoded.gz', 'InsuranceQA.question.anslabel.raw.1000.pool.solr.train.encoded.gz', 'InsuranceQA.question.anslabel.raw.1000.pool.solr.valid.encoded.gz', 'InsuranceQA.question.anslabel.raw.1500.pool.solr.test.encoded.gz', 'InsuranceQA.question.anslabel.raw.1500.pool.solr.train.encoded.gz', 'InsuranceQA.question.anslabel.raw.1500.pool.solr.valid.encoded.gz', 'InsuranceQA.question.anslabel.raw.500.pool.solr.test.encoded.gz', 'InsuranceQA.question.anslabel.raw.500.pool.solr.train.encoded.gz', 'InsuranceQA.question.anslabel.raw.500.pool.solr.valid.encoded.gz', 'InsuranceQA.question.anslabel.raw.encoded.gz', 'InsuranceQA.question.anslabel.toke

In [4]:
bt = load_dataset("bitext/Bitext-insurance-llm-chatbot-training-dataset", split="train")
print(bt.column_names, len(bt))
need = {"instruction", "intent", "category", "tags", "response"}
assert need <= set(bt.column_names), bt.column_names
bitext = [{"source": "bitext", "category": r["category"], "intent": r["intent"],
           "tags": r["tags"], "instruction": r["instruction"], "response": r["response"]}
          for r in bt]
print(bitext[0])

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

bitext-insurance-llm-chatbot-training-da(…): reconstructing file:   0%|          |  0.00B / 27.2MB            

bitext-insurance-llm-chatbot-training-da(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/39000 [00:00<?, ? examples/s]

['instruction', 'intent', 'category', 'tags', 'response'] 39000
{'source': 'bitext', 'category': 'AUTO_INSURANCE', 'intent': 'information_auto_insurance', 'tags': 'BCLPWZ', 'instruction': "I'd like to see my fucking auto insurance , I need assistance", 'response': 'To retrieve the details of your auto insurance, please adhere to the following instructions:\n\n1. Access your account by navigating to {{WEBSITE_URL}}.\n2. Proceed to the {{MY_POLICIES_SECTION}} section within your profile.\n3. Select the auto insurance policy that you want to review.\n\nThis process will grant you comprehensive access to your auto insurance information. Should you require any additional support, do not hesitate to contact me.'}


In [5]:
bench = []
for split in ["train", "validation", "test"]:
    p = hf_hub_download("pashas/insurance-ai-reliability-benchmark",
                        f"data/{split}.jsonl", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                rec = {"source": "insurance_benchmark"}
                rec.update({k: v for k, v in r.items() if k != "source"})
                bench.append(rec)
print(len(bench), "benchmark records (expected 510)")
print(json.dumps(bench[0], ensure_ascii=False)[:1200])

train.jsonl:   0%|          | 0.00/419k [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/90.5k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/90.2k [00:00<?, ?B/s]

510 benchmark records (expected 510)
{"source": "insurance_benchmark", "id": "claim_status_015", "category": "claim_status", "subcategory": "status_denied", "difficulty": "hard", "input": {"customer_message": "Can you check on claim CLM-728492? I'm getting worried.", "context": {"policy_type": "workers_comp", "policy_status": "active", "customer_tier": "high_value", "time_of_day": "weekend", "channel": "email", "state": "FL"}, "conversation_history": []}, "expected_output": {"intent": "check_claim_status", "routing_decision": "human_handoff", "required_actions": ["lookup_claim", "explain_denial_reason", "explain_appeal_process", "transfer_to_supervisor"], "priority": "high", "compliance_flags": ["escalation_risk"], "expected_response_elements": ["status_update", "next_steps", "timeline"]}, "evaluation_criteria": {"intent_accuracy": true, "routing_correctness": true, "action_completeness": 1.0, "response_quality_rubric": {"status": "Must provide current claim status", "timeline": "Must 

In [6]:
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = False,
    full_finetuning = False,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16, lora_alpha = 16, lora_dropout = 0,
    bias = "none", random_state = SEED,
)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")
tok = getattr(tokenizer, "tokenizer", tokenizer)

==((====))==  Unsloth 2026.10.11: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

In [7]:
def chat_text(rec):
    convo = [{"role": "user", "content": PROMPT},
             {"role": "assistant", "content": json.dumps(rec, ensure_ascii=False)}]
    return tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)

def select(recs, n):
    recs = recs[:]; random.shuffle(recs)
    kept, dropped = [], 0
    for i in range(0, len(recs), 256):
        chunk = recs[i:i+256]
        texts = [chat_text(r) for r in chunk]
        lens = [len(x) for x in tok(texts, add_special_tokens=False)["input_ids"]]
        for r, t, l in zip(chunk, texts, lens):
            if l <= MAX_EX_TOKENS: kept.append((r, t))
            else: dropped += 1
        if n and len(kept) >= n: break
    return (kept[:n] if n else kept), dropped

k1, d1 = select(iqa, N_IQA)
k2, d2 = select(bitext, N_BITEXT)
k3, d3 = select(bench, None)
print(f"too long, dropped -> iqa:{d1} bitext:{d2} bench:{d3}")

pairs = k1 + k2 + k3 * BENCH_REPEAT
random.shuffle(pairs)
train_records = [r for r, _ in pairs]          # kept for validation later
dataset = Dataset.from_dict({"text": [t for _, t in pairs]})

print(pd.Series([r["source"] for r in train_records]).value_counts(normalize=True).round(3))
print(len(dataset), "training examples")
print(dataset[0]["text"][:700])

too long, dropped -> iqa:2 bitext:0 bench:0
bitext                 0.398
insuranceqa            0.398
insurance_benchmark    0.203
Name: proportion, dtype: float64
5020 training examples
<bos><|turn>user
Generate synthetic data.<turn|>
<|turn>model
{"source": "insurance_benchmark", "id": "error_recovery_033", "category": "error_recovery", "subcategory": "incorrect_premium", "difficulty": "hard", "input": {"customer_message": "My premium is way higher than what I was quoted.", "context": {"policy_type": "homeowners", "policy_status": "active", "customer_tier": "new_customer", "time_of_day": "after_hours", "channel": "email", "state": "FL"}, "conversation_history": []}, "expected_output": {"intent": "resolve_premium_discrepancy", "routing_decision": "human_handoff", "required_actions": ["verify_identity", "compare_quote_to_policy", "document_discrepancy", "escalate_to_underwri


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    train_dataset = dataset, eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10 if SMOKE_TEST else 30,
        num_train_epochs = 1,
        max_steps = 30 if SMOKE_TEST else -1,      # -1 = one full epoch
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = SEED,
        report_to = "none",
    ),
)
trainer = train_on_responses_only(trainer)     # loss only on the JSON record, not the prompt
trainer_stats = trainer.train()

In [8]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",

        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,

        warmup_steps=5,
        max_steps=200,
        learning_rate=1e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=SEED,
        report_to="none",
        output_dir="outputs",

        gradient_checkpointing=True,
        save_strategy="steps",
        save_steps=50,
    ),
)

trainer = train_on_responses_only(trainer)
trainer_stats = trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5020 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Unsloth: Auto-detected instruction_part = '<|turn>user\n' and response_part = '<|turn>model\n'


Map (num_proc=2):   0%|          | 0/5020 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,020 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)


Step,Training Loss
5,4.116430
10,3.077714
15,2.093922
20,2.209913
25,1.925456
30,1.751546
35,1.599498
40,1.625745
45,1.473295
50,1.196047


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-200/tokenizer_config.json.


In [9]:
model.save_pretrained("gemma_insurance_lora")
tokenizer.save_pretrained("gemma_insurance_lora")

from google.colab import drive; drive.mount('/content/drive')
!cp -r gemma_insurance_lora /content/drive/MyDrive/

Unsloth: Restored added_tokens_decoder metadata in gemma_insurance_lora/tokenizer_config.json.


Mounted at /content/drive


In [11]:
FastModel.for_inference(model)

REQUIRED = {}
for r in train_records:
    REQUIRED.setdefault(r["source"], set(r.keys()))

def parse(txt):
    txt = txt.strip()
    txt = re.sub(r"^```(?:json)?\s*|\s*```$", "", txt)
    try: rec = json.loads(txt)
    except Exception: return None
    if isinstance(rec, dict) and rec.get("source") in REQUIRED and REQUIRED[rec["source"]] <= set(rec.keys()):
        return rec
    return None

def generate_records(n_total=300, per_call=8, max_new_tokens=850, temperature=1.0, top_p=0.95, top_k=64):
    msgs = [{"role": "user", "content": PROMPT}]
    # Use tok.apply_chat_template instead of tokenizer.apply_chat_template to avoid processor wrapping errors
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                     tokenize=True, return_dict=True).to("cuda")
    plen = inputs["input_ids"].shape[1]
    out, bad, tries = [], 0, 0
    while len(out) < n_total and tries < 4 * n_total // max(per_call, 1) + 20:
        tries += 1
        try:
            gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                                 temperature=temperature, top_p=top_p, top_k=top_k,
                                 num_return_sequences=per_call)
        except Exception as e:
            print("generate failed, falling back to 1 per call:", repr(e)[:200])
            per_call = 1; continue
        for g in gen:
            rec = parse(tok.decode(g[plen:], skip_special_tokens=True))
            if rec is None: bad += 1
            else: out.append(rec)
    print(f"valid={len(out)}  invalid={bad}  validity rate={len(out)/max(len(out)+bad,1):.1%}")
    return out[:n_total]

synth = generate_records(300)
for s in synth[:3]: print(json.dumps(s, ensure_ascii=False)[:600], "\n")


valid=305  invalid=7  validity rate=97.8%
{"source": "insuranceqa", "domain": "auto-insurance", "question": "Do My Own Auto Insurance Make Sense?", "answer": "Yes, the majority of personal car insurance policyholders understand that an independent or discount carrier will be better on the premium than a regulated carrier. There will be no significant negative consequences of doing that. Only a handful of persons are who know not to do it. In many regions you do not need to buy car insurance if you do not own a car, then a credit card can cover it. For an 18 year old with no history you might be able to get a discounted rate than payin 

{"source": "insuranceqa", "domain": "medicare-insurance", "question": "Which Medicare Plan Is Best?", "answer": "The right Medicare Plan depends on your individual circumstances. Each plan caters to a different set of priorities, so it's important to select the one that most appropriately meets your needs. When choosing, analyze the plan in detail - Med

In [12]:
from scipy.stats import ks_2samp

def flatten(records):
    df = pd.json_normalize(records, sep=".")
    for c in df.columns:
        df[c] = df[c].map(lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v)
    return df

def tvd(a, b):
    pa, pb = a.astype(str).value_counts(normalize=True), b.astype(str).value_counts(normalize=True)
    idx = pa.index.union(pb.index)
    return 0.5 * (pa.reindex(idx, fill_value=0) - pb.reindex(idx, fill_value=0)).abs().sum()

real_f, synth_f = flatten(train_records), flatten(synth)

print("== source mix ==")
print(pd.concat([real_f["source"].value_counts(normalize=True).rename("real"),
                 synth_f["source"].value_counts(normalize=True).rename("synthetic")], axis=1).round(3))

for src in sorted(real_f["source"].unique()):
    R = real_f[real_f.source == src].dropna(axis=1, how="all")
    S = synth_f[synth_f.source == src]
    print(f"\n===== {src}: real={len(R)} synthetic={len(S)} =====")
    if len(S) < 5: print("too few synthetic records"); continue
    rl = [len(json.dumps(r)) for r in train_records if r["source"] == src]
    sl = [len(json.dumps(r)) for r in synth if r["source"] == src]
    ks = ks_2samp(rl, sl)
    print(f"record length: real mean={np.mean(rl):.0f} synth mean={np.mean(sl):.0f}  KS={ks.statistic:.3f}")
    for col in R.columns:
        if col == "source" or col not in S.columns: continue
        if pd.api.types.is_numeric_dtype(R[col]) and R[col].nunique() > 1:
            ks = ks_2samp(R[col].dropna(), S[col].dropna().astype(float))
            print(f"  [numeric] {col}: real mean={R[col].mean():.3g} synth mean={S[col].astype(float).mean():.3g} KS={ks.statistic:.3f}")
        elif 1 < R[col].nunique() <= 60:
            print(f"  [categorical] {col}: TVD={tvd(R[col], S[col]):.3f}   (0 = identical)")

real_set = {json.dumps(r, sort_keys=True) for r in train_records}
synth_s  = [json.dumps(r, sort_keys=True) for r in synth]
print(f"\nexact copies of training rows: {sum(s in real_set for s in synth_s)}/{len(synth)}")
print(f"duplicates inside synthetic:   {len(synth_s) - len(set(synth_s))}/{len(synth)}")

# ---- export ----
synth_f.to_csv("synthetic_insurance_all.csv", index=False)
with open("synthetic_insurance_all.json", "w", encoding="utf-8") as f:
    json.dump(synth, f, ensure_ascii=False, indent=2)
for src in synth_f["source"].unique():
    synth_f[synth_f.source == src].dropna(axis=1, how="all").to_csv(f"synthetic_{src}.csv", index=False)

from google.colab import files
for fn in ["synthetic_insurance_all.csv", "synthetic_insurance_all.json"]:
    files.download(fn)

== source mix ==
                      real  synthetic
source                               
bitext               0.398      0.387
insuranceqa          0.398      0.430
insurance_benchmark  0.203      0.183

===== bitext: real=2000 synthetic=116 =====
record length: real mean=802 synth mean=769  KS=0.103
  [categorical] category: TVD=0.182   (0 = identical)
  [categorical] intent: TVD=0.699   (0 = identical)

===== insurance_benchmark: real=1020 synthetic=55 =====
record length: real mean=1209 synth mean=1207  KS=0.183
  [categorical] category: TVD=0.359   (0 = identical)
  [categorical] difficulty: TVD=0.070   (0 = identical)
  [categorical] input.context.policy_type: TVD=0.351   (0 = identical)
  [categorical] input.context.customer_tier: TVD=0.085   (0 = identical)
  [categorical] input.context.time_of_day: TVD=0.159   (0 = identical)
  [categorical] input.context.channel: TVD=0.025   (0 = identical)
  [categorical] input.context.state: TVD=0.400   (0 = identical)
  [categorical] in

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>